It’s the Handle class.

<h2>A use case for handles: setting up</h2>
Let’s say we hold a number of different bonds. In the first part of the code, we build four of them and collect them in a list. In the interest of brevity, they’re all zero-coupon bonds with different maturity dates; note, though, that we could have easily mixed different kinds of bonds there.

In [2]:
import QuantLib as ql

In [16]:
value_date = ql.Date(7, ql.March, 2023)
ql.Settings.instance().evaluationDate = value_date

settlement_days = 3
face_amount = 100.0
calendar = ql.TARGET()

bonds = [
    ql.ZeroCouponBond(
        settlement_days,
        calendar,
        face_amount,
        ql.Date(10, ql.February, 2025),
    ),
    ql.ZeroCouponBond(
        settlement_days,
        calendar,
        face_amount,
        ql.Date(26, ql.August, 2030),
    ),
    ql.ZeroCouponBond(
        settlement_days, calendar, face_amount, ql.Date(13, ql.May, 2024)
    ),
    ql.ZeroCouponBond(
        settlement_days,
        calendar,
        face_amount,
        ql.Date(15, ql.August, 2040),
    ),
]

Then, let’s assume that the bonds were issued by two different entities, and that we have discount curves available for both. They might come from an external system, or you might have bootstrapped or fitted them using some other facility provided by QuantLib. In this example, we build them using the ZeroCurve class, which interpolates a set of given zero rates at different dates; a real system would provide us with a lot more nodes but, again, I tried to keep the code short. As for bonds, the specific class doesn’t matter; any class inherited from the base YieldTermStructure class would do.

In [17]:
curve_nodes = [
    value_date,
    value_date + ql.Period(10, ql.Years),
    value_date + ql.Period(20, ql.Years),
]
curve_rates_1 = [0.015, 0.025, 0.028]
curve_rates_2 = [0.005, 0.009, 0.012]

issuer_curve_1 = ql.ZeroCurve(curve_nodes, curve_rates_1, ql.Actual360())
issuer_curve_2 = ql.ZeroCurve(curve_nodes, curve_rates_2, ql.Actual360())

And here comes the step which might seem unnecessary: instead of using the curves directly, we wrap them in handles, namely, we pass each curve to an instance of RelinkableYieldTermStructureHandle. Yes, it’s a mouthful.

In [18]:
issuer_handle_1 = ql.RelinkableYieldTermStructureHandle(issuer_curve_1)
issuer_handle_2 = ql.RelinkableYieldTermStructureHandle(issuer_curve_2)

To complete our setup, we pass each of the handles to an instance of DiscountingBondEngine, which can calculate bond prices by discounting their coupons according to the given curve. Assuming that the first two bonds come from the first issuer and the second two from the other, we set the relevant engine to each bond; and finally, we can output their prices.

In [19]:
engine_1 = ql.DiscountingBondEngine(issuer_handle_1)
engine_2 = ql.DiscountingBondEngine(issuer_handle_2)

bonds[0].setPricingEngine(engine_1)
bonds[1].setPricingEngine(engine_1)
bonds[2].setPricingEngine(engine_2)
bonds[3].setPricingEngine(engine_2)

for b in bonds:
    print(f"{b.cleanPrice():.4f}")

96.7459
84.3484
99.3479
81.9757


<h2>The benefits of using handles</h2>

Now, let’s suppose you want to create a modified curve in order to simulate what happens if the credit rating of the first issuer worsens; or maybe you want to change the set of instruments on which you’re fitting one of your curves, or again your system tells you that one of the curves has changed.

Whatever the reason, we can create a new curve and pass it to the linkTo method of the corresponding handle: in this example, to simulate a worsening of the credit, we use a ZeroSpreadedTermStructure instance which adds a spread on top of the existing curve. After doing this, we ask the bonds for their prices, and the first two bonds will return updated figures. No need to tell the bonds or the engines that their discount curve has changed; the library takes care of that.

In [20]:
shifted_curve_1 = ql.ZeroSpreadedTermStructure(
    ql.YieldTermStructureHandle(issuer_curve_1),
    ql.QuoteHandle(ql.SimpleQuote(0.001))
)

issuer_handle_1.linkTo(shifted_curve_1)

for b in bonds:
    print(f"{b.cleanPrice():.4f}")

96.5572
83.7121
99.3479
81.9757


This, quite simply, is the reason for the existence of handles and the central role they play in the design of QuantLib: the ability to manage different term structures, quotes, and any kind of market dependencies without having to update each instrument explicitly or without recreating the engine.